# Stochastic Simulation — Drill Exercises (Exam-Focused)

These exercises are drawn directly from 2023–2025 past papers. Cover them until you can reproduce every answer cold, with no notes.

**Exam format:** 4 questions × 20 marks = 80 marks total. 80% = 64 marks.

**Priority order:** Q1 (sampling) → Q2 (IS/MC) → Q3 (MCMC) → Q4 (SSM/PF).

---

## Table of Contents
1. [Block A — Inversion Sampling & Transformations (Q1 fodder)](#block-a)
2. [Block B — Rejection Sampling: M computation, acceptance rate, optimal proposal (Q1 fodder)](#block-b)
3. [Block C — Monte Carlo Estimators: unbiasedness, variance (Q2 fodder)](#block-c)
4. [Block D — Importance Sampling: estimator, variance derivation, optimal proposal (Q2 fodder)](#block-d)
5. [Block E — Bayesian Conjugate Updates (Q2/Q3 fodder)](#block-e)
6. [Block F — Metropolis-Hastings: pseudocode, detailed balance proof, rejection probability (Q3 fodder)](#block-f)
7. [Block G — Langevin Algorithms: ULA limiting distribution, MALA proposal (Q3 fodder)](#block-g)
8. [Block H — Fisher's Identity & SOUL / EM (Q2/Q5 fodder)](#block-h)
9. [Block I — State-Space Models & Particle Filter (Q4 fodder)](#block-i)
10. [Block J — SIR Smoother & Gradient Ascent for $\theta$ (Q4 fodder)](#block-j)

---
<a id='block-a'></a>
## Block A — Inversion Sampling & Transformations

These are standard 2–5 mark questions that open Q1 every year.

---

### A1. Inversion method — generic proof *(2023 Q1a, 2024 Q1a, 2025 Q1a)*

**Question.** Let $U \sim \text{Unif}(0,1)$ and $Y = F_X^{-1}(U)$. Prove that $p_Y = p_X$.

**Answer.**
$$P(Y \leq y) = P(F_X^{-1}(U) \leq y) = P(U \leq F_X(y)) = F_X(y).$$
Hence $Y \sim p_X$. $\square$

**Variant (2025 Q1a).** Show that if $X \sim p_X$ then $Y = F_Y^{-1}(F_X(X))$ gives $Y \sim p_Y$.

**Answer.** $U = F_X(X) \sim \text{Unif}(0,1)$ (probability integral transform). Then $Y = F_Y^{-1}(U)$, so by the above $Y \sim p_Y$. $\square$

---

### A2. Inversion for a specific distribution *(2023 Q1a, 2024 Q1a)*

**Question (Rayleigh, 2023).** The Rayleigh density is $p(x) = \frac{x}{\sigma^2}\exp\!\left(-\frac{x^2}{2\sigma^2}\right)$, $x \geq 0$. 
(i) Show $F_X(x) = 1 - \exp\!\left(-\frac{x^2}{2\sigma^2}\right)$. 
(ii) Invert to get $F_X^{-1}(u) = \sqrt{-2\sigma^2 \log(1-u)}$. 
(iii) Write the sampler pseudocode. 
(iv) Justify using $U \sim \text{Unif}(0,1)$ directly: $X = \sqrt{-2\sigma^2 \log U}$.

**Answers.**

(i) $F_X(x) = \int_0^x \frac{t}{\sigma^2}e^{-t^2/(2\sigma^2)}dt$. Substitute $v = t^2/(2\sigma^2)$, $dv = t/\sigma^2\,dt$:
$$F_X(x) = \int_0^{x^2/(2\sigma^2)} e^{-v}\,dv = 1 - e^{-x^2/(2\sigma^2)}.$$

(ii) Set $u = 1 - e^{-x^2/(2\sigma^2)}$, so $e^{-x^2/(2\sigma^2)} = 1-u$, giving $F_X^{-1}(u) = \sqrt{-2\sigma^2\log(1-u)}$.

(iii) Pseudocode:
```
sample U ~ Unif(0,1)
return X = sqrt(-2*sigma^2 * log(1 - U))
```

(iv) Since $1-U \sim \text{Unif}(0,1)$ when $U \sim \text{Unif}(0,1)$, we can replace $1-U$ by $U$, giving the same distribution.

---

### A3. Box-Muller / Multivariate change of variables *(2024 Q1b, 2025 Q1b)*

**Question (2024).** Let $U_1 \sim \text{Unif}(0,1)$, $U_2 \sim \text{Unif}(-\pi,\pi)$, and 
$$X_1 = \cos(U_2)\sqrt{-2\log U_1}, \quad X_2 = \sin(U_2)\sqrt{-2\log U_1}.$$
Show $X_1, X_2 \overset{\text{iid}}{\sim} \mathcal{N}(0,1)$.

**Answer.** Write $R = \sqrt{X_1^2+X_2^2}$, $\Theta = \arctan(X_2/X_1)$. The inverse map is $U_1 = e^{-(X_1^2+X_2^2)/2}$, $U_2 = \arctan(X_2/X_1)$. The Jacobian $|J| = \frac{1}{2\pi}e^{(X_1^2+X_2^2)/2}\cdot\frac{1}{X_1^2+X_2^2}\cdot\frac{1}{2}$ … more cleanly: compute the joint density of $(U_1, U_2)$ (uniform on $(0,1)\times(-\pi,\pi)$, so $p_{U_1,U_2} = \frac{1}{2\pi}$) and apply the change-of-variables formula:
$$p_{X_1,X_2}(x_1,x_2) = p_{U_1,U_2}(u_1(x_1,x_2), u_2(x_1,x_2))\cdot|J|$$
where $u_1 = e^{-(x_1^2+x_2^2)/2}$, $u_2 = \arctan(x_2/x_1)$, and the Jacobian determinant is:
$$|J| = \left|\frac{\partial(u_1,u_2)}{\partial(x_1,x_2)}\right|^{-1}.$$
Compute: $\frac{\partial u_1}{\partial x_1} = -x_1 e^{-r^2/2}$, $\frac{\partial u_1}{\partial x_2} = -x_2 e^{-r^2/2}$, $\frac{\partial u_2}{\partial x_1} = -x_2/r^2$, $\frac{\partial u_2}{\partial x_2} = x_1/r^2$ where $r^2 = x_1^2+x_2^2$. Determinant of forward Jacobian:
$$\det = (-x_1 e^{-r^2/2})(x_1/r^2) - (-x_2 e^{-r^2/2})(-x_2/r^2) = -e^{-r^2/2}.$$
So $|J|^{-1} = e^{r^2/2}$ (use absolute value). Therefore:
$$p_{X_1,X_2}(x_1,x_2) = \frac{1}{2\pi} \cdot e^{r^2/2} \cdot e^{-r^2/2} \cdot \frac{1}{r^2}\cdot r^2 = \frac{1}{2\pi}e^{-(x_1^2+x_2^2)/2}$$
Wait — redo carefully. The inverse Jacobian $|\partial(x_1,x_2)/\partial(u_1,u_2)|$: from $x_1 = \cos(u_2)\sqrt{-2\log u_1}$, $x_2 = \sin(u_2)\sqrt{-2\log u_1}$, compute:
$$\frac{\partial x_1}{\partial u_1} = \frac{-\cos(u_2)}{u_1\sqrt{-2\log u_1}}, \quad \frac{\partial x_1}{\partial u_2} = -\sin(u_2)\sqrt{-2\log u_1}$$
$$\frac{\partial x_2}{\partial u_1} = \frac{-\sin(u_2)}{u_1\sqrt{-2\log u_1}}, \quad \frac{\partial x_2}{\partial u_2} = \cos(u_2)\sqrt{-2\log u_1}$$
Determinant:
$$\det J = \frac{-\cos^2(u_2)}{u_1} - \frac{\sin^2(u_2)}{u_1} \cdot\frac{-2\log u_1}{-2\log u_1}\cdot(-1) \Rightarrow |\det J| = \frac{-2\log u_1}{u_1} \cdot(-1) = \frac{-2\log u_1}{u_1}.$$
So $|\det J| = \frac{-2\log u_1}{u_1}$ (positive since $u_1 \in (0,1)$ so $\log u_1 < 0$). Substituting $u_1 = e^{-(x_1^2+x_2^2)/2}$:
$$|\det J| = \frac{x_1^2+x_2^2}{e^{-(x_1^2+x_2^2)/2}}.$$
Therefore:
$$p_{X_1,X_2}(x_1,x_2) = \frac{1}{2\pi} \cdot \frac{e^{-(x_1^2+x_2^2)/2}}{x_1^2+x_2^2} \cdot \text{[need extra factor]}$$
Actually the density transforms as $p_X(x) = p_U(u)|\det \partial u/\partial x|$. We have $p_{U_1,U_2} = \frac{1}{2\pi}$, so:
$$p_{X_1,X_2}(x_1,x_2) = \frac{1}{2\pi}\cdot|\det J_{u\leftarrow x}| = \frac{1}{2\pi}\cdot e^{-(x_1^2+x_2^2)/2} = \frac{1}{\sqrt{2\pi}}e^{-x_1^2/2}\cdot\frac{1}{\sqrt{2\pi}}e^{-x_2^2/2}.$$
This factors into two independent $\mathcal{N}(0,1)$ densities. $\square$

**Key formula to memorise:** The multivariate change-of-variables formula is
$$p_Y(y) = p_X(g^{-1}(y))\,|\det J_{g^{-1}}(y)|$$
where $y = g(x)$ and $J_{g^{-1}}$ is the Jacobian of the inverse map.

---
<a id='block-b'></a>
## Block B — Rejection Sampling

---

### B1. Pseudocode and acceptance rate *(every year)*

**Question.** For target $p(x)$ and proposal $q(x)$ with $p(x) \leq Mq(x)$ for all $x$:
(a) Give pseudocode. 
(b) Show acceptance rate $\hat{a} = 1/M$.

**Answer (a) — Pseudocode:**
```
repeat:
    sample X ~ q(x)
    sample U ~ Unif(0, 1)
    if U <= p(X) / (M * q(X)):
        return X
```

**Answer (b) — Acceptance rate:**
$$\hat{a} = P\!\left(U \leq \frac{p(X)}{Mq(X)}\right) = \int q(x)\int_0^{p(x)/(Mq(x))} du\,dx = \int q(x)\cdot\frac{p(x)}{Mq(x)}\,dx = \frac{1}{M}\int p(x)\,dx = \frac{1}{M}.$$

---

### B2. Unnormalised target — acceptance rate *(2024 Q1c)*

**Question.** Now suppose only $\bar{p}(x) \propto p(x)$ is available, with $\bar{p}(x) \leq \bar{M}q(x)$. Show acceptance rate is $\hat{a} = Z/\bar{M}$ where $Z = \int \bar{p}(x)dx$.

**Answer.** The acceptance condition is $U \leq \bar{p}(X)/(\bar{M}q(X))$. By the same calculation:
$$\hat{a} = \int q(x)\cdot\frac{\bar{p}(x)}{\bar{M}q(x)}\,dx = \frac{1}{\bar{M}}\int \bar{p}(x)\,dx = \frac{Z}{\bar{M}}.$$

---

### B3. Finding $M$ and the optimal proposal *(2023 Q1b, 2024 Q1d, 2025 Q1c)*

**Template.** Given $p(x)$ and $q(x)$:
1. Form $h(x) = p(x)/q(x)$.
2. Maximise $h(x)$ over $x$: solve $h'(x^\star) = 0$, check it's a max.
3. Set $M = h(x^\star) = \sup_x p(x)/q(x)$.
4. Acceptance rate $= 1/M$. To optimise over proposal parameter $\lambda$: maximise $1/M(\lambda)$, i.e. minimise $M(\lambda)$.

**Example (2025 Q1c).** $p(x) \propto x^2 e^{-x/2}$, $q_\lambda(x) = \lambda e^{-\lambda x}$.

(i) $M(\lambda) = \sup_x \frac{x^2 e^{-x/2}}{\lambda e^{-\lambda x}} = \sup_x \frac{x^2}{\lambda} e^{(\lambda-1/2)x}$. For this to be finite, need $\lambda < 1/2$. Maximise: $\frac{d}{dx}[x^2 e^{(\lambda-1/2)x}] = 0 \Rightarrow 2x + (\lambda - 1/2)x^2 = 0 \Rightarrow x^\star = \frac{-2}{\lambda - 1/2} = \frac{2}{1/2-\lambda} = \frac{4}{1-2\lambda}$.
$$M(\lambda) = \frac{(x^\star)^2}{\lambda}\exp\!\left((\lambda - 1/2)x^\star\right).$$

(ii) To find $\lambda^\star$: differentiate $\log(1/M) = \log M^{-1}$ w.r.t. $\lambda$ and set to zero, or equivalently differentiate $\log M$ w.r.t. $\lambda$. After algebra: $\lambda^\star = 1/4$.

(iii) Acceptance rate at $\lambda^\star = 1/4$: $x^\star = 4/(1-1/2) = 8$.
$$M(1/4) = \frac{64}{1/4}e^{(1/4-1/2)\cdot 8} = 256 \cdot e^{-2}.$$
But $p(x) = x^2 e^{-x/2}/Z$ where $Z = \int_0^\infty x^2 e^{-x/2}dx = 16$ (Gamma function: $\int_0^\infty x^2 e^{-x/2}dx = 2^3\Gamma(3) = 16$). 
Acceptance rate $= Z/M(1/4)$ (since $p$ is normalised... wait, $p(x) = \bar{p}(x)/Z$ so if we use $\bar{p}(x) = x^2e^{-x/2}$, acceptance $= Z/\bar{M}$ where $\bar{M} = 256e^{-2}$): $\hat{a} = 16/(256e^{-2}) = e^2/16$.

(v) Distribution of $N$ (accepted samples from $K$ iterations): each iteration independently accepts with probability $\hat{a}$, so $N \sim \text{Binomial}(K, \hat{a})$. Thus $E[N] = K\hat{a}$.

**Key fact to memorise:** $N \sim \text{Binomial}(K, 1/M)$ when $p$ is normalised.

---

### B4. Weibull inversion + CDF derivation *(2024 Q1a)*

**Question.** $p_X(x) = kx^{k-1}e^{-x^k}$, $x \geq 0$. Show $F_X(x) = 1-e^{-x^k}$, find $F_X^{-1}$, write sampler.

**Answer.**
$$F_X(x) = \int_0^x k t^{k-1}e^{-t^k}dt \overset{v=t^k}{=} \int_0^{x^k} e^{-v}dv = 1 - e^{-x^k}.$$
$F_X^{-1}(u) = (-\log(1-u))^{1/k}$. Sampler: draw $U \sim \text{Unif}(0,1)$, return $X = (-\log(1-U))^{1/k}$.

---
<a id='block-c'></a>
## Block C — Monte Carlo Estimators

---

### C1. MC estimator: definition, unbiasedness, variance *(2023 Q2a)*

**Question.** Define the MC estimator for $I = \int \varphi(x)p(x)dx$ from $X^{(1)},\ldots,X^{(N)} \overset{\text{iid}}{\sim} p$. Prove unbiasedness. State the variance.

**Answer.**
$$\hat{\varphi}^N_{\text{MC}} = \frac{1}{N}\sum_{i=1}^N \varphi(X^{(i)}).$$

**Unbiasedness:**
$$E[\hat{\varphi}^N_{\text{MC}}] = \frac{1}{N}\sum_{i=1}^N E[\varphi(X^{(i)})] = \frac{1}{N}\cdot N\int \varphi(x)p(x)dx = I.$$

**Variance:**
$$\text{Var}(\hat{\varphi}^N_{\text{MC}}) = \frac{1}{N}\text{Var}_{p}(\varphi(X)) = \frac{1}{N}\left(\int \varphi(x)^2 p(x)dx - I^2\right).$$

---

### C2. MC variance — explicit computation *(2024 Q2b, 2023 Q2a)*

**Template.** $\text{Var}(\hat{\varphi}^N_{\text{MC}}) = \frac{1}{N}\left[\int \varphi(x)^2 p(x)dx - \left(\int \varphi(x)p(x)dx\right)^2\right]$.

To compute: (1) compute $\int \varphi(x)^2 p(x)dx$ via substitution/integration by parts, (2) subtract the squared mean.

**Example (2024 Q2b).** $p(x) = xe^{-x^2/2}$, $\varphi(x) = x$, true mean $= \sqrt{\pi/2}$.
$$\int x^2 \cdot xe^{-x^2/2}dx = \int_0^\infty x^3 e^{-x^2/2}dx.$$
Let $v = x^2/2$: $= \int_0^\infty 2v e^{-v}dv = 2\Gamma(2) = 2$. So:
$$\text{Var} = \frac{1}{N}\left(2 - \frac{\pi}{2}\right).$$

---
<a id='block-d'></a>
## Block D — Importance Sampling

---

### D1. IS estimator and variance *(2023 Q2b, 2024 Q2a)*

**Question.** Define the IS estimator for $I = \int \varphi(x)p(x)dx$ using proposal $q(x)$. Prove unbiasedness. State and derive the variance.

**IS estimator:**
$$\hat{\varphi}^N_{\text{IS}} = \frac{1}{N}\sum_{i=1}^N \frac{p(X^{(i)})}{q(X^{(i)})}\varphi(X^{(i)}), \quad X^{(i)} \overset{\text{iid}}{\sim} q.$$

**Unbiasedness:**
$$E_q\!\left[\frac{p(X)}{q(X)}\varphi(X)\right] = \int \frac{p(x)}{q(x)}\varphi(x)\cdot q(x)dx = \int \varphi(x)p(x)dx = I.$$

**Variance:**
$$\text{Var}(\hat{\varphi}^N_{\text{IS}}) = \frac{1}{N}\left[\int \frac{\varphi(x)^2 p(x)^2}{q(x)}dx - I^2\right].$$

**Finite variance condition:** $\text{supp}(p) \subseteq \text{supp}(q)$ and $\int \frac{\varphi(x)^2 p(x)^2}{q(x)}dx < \infty$.

---

### D2. IS variance — explicit computation *(2024 Q2a, 2023 Q2c)*

**Example (2024 full Q2a).** $p(x) = xe^{-x^2/2}$ (Rayleigh), $q_\lambda(x) = \frac{x}{\lambda^2}e^{-x^2/(2\lambda^2)}$ (Rayleigh with scale $\lambda$), $\varphi(x) = x$.

$$\text{Var} = \frac{1}{N}\left[\int_0^\infty \frac{x^2 (xe^{-x^2/2})^2}{(x/\lambda^2)e^{-x^2/(2\lambda^2)}}dx - \frac{\pi}{2}\right]
= \frac{1}{N}\left[\lambda^2\int_0^\infty x^4 e^{-x^2(1-1/\lambda^2)}dx - \frac{\pi}{2}\right].$$

After integration by parts (let $v = x^2(1-1/\lambda^2)$), the result is:
$$\text{Var}(\hat{\varphi}^N_{\text{IS}}) = \frac{1}{N}\left[\frac{2\lambda^6}{(2\lambda^2-1)^2} - \frac{\pi}{2}\right].$$

**Optimal $\lambda^\star$:** differentiate w.r.t. $\lambda^2$ (let $s = \lambda^2$):
$$\frac{d}{ds}\frac{2s^3}{(2s-1)^2} = 0 \Rightarrow 6s^2(2s-1)^2 - 2s^3\cdot 2(2s-1)\cdot 2 = 0 \Rightarrow 6s^2(2s-1) - 8s^3 = 0$$
$$\Rightarrow s(12s - 6 - 8s) = 0 \Rightarrow 4s = 6 \Rightarrow s = 3/2 \Rightarrow \lambda^\star = \sqrt{3/2}.$$

---

### D3. Optimal IS proposal — mean shift *(2023 Q2c, 2025 Q2c)*

**Template.** Given $p = \mathcal{N}(0,1)$, $q_\mu = \mathcal{N}(\mu, \sigma_q^2)$, $\varphi(x) = \mathbf{1}_{x \geq K}$.
Minimise $\text{Var} = \frac{1}{N}\left[\int \frac{\varphi(x)^2 p(x)^2}{q_\mu(x)}dx - I^2\right]$ over $\mu$. Ignore $I^2$ (constant), so minimise $\int_{K}^\infty \frac{p(x)^2}{q_\mu(x)}dx$.

**Example (2023 Q2c).** $p = \mathcal{N}(0,1)$, $q_\mu = \mathcal{N}(\mu, 1/2)$.
$$\int_K^\infty \frac{e^{-x^2}}{(2\pi\cdot 1/2)^{-1/2} e^{-(x-\mu)^2/1}}dx = \text{const}\cdot \int_K^\infty e^{-x^2+(x-\mu)^2}dx = \text{const}\cdot\int_K^\infty e^{\mu^2 - 2\mu x}dx.$$
This is finite iff the integrand decays, which requires examining the exponent. Differentiating under the integral sign w.r.t. $\mu$ and setting to zero yields $\mu^\star = y_0/2$ in the 2023 version (with Gaussian likelihood). 

**Key takeaway:** optimal $\mu$ shifts mass to where $\varphi(x)p(x)/q_\mu(x)$ is concentrated, i.e., near the important region.

---

### D4. Self-normalised IS (unnormalised weights) *(2025 Q2a, 2024 Q4a)*

**Question.** Suppose only $\bar{p}(x)$ is known ($p(x) = \bar{p}(x)/Z$). Derive the self-normalised IS estimator.

**Answer.** Define unnormalised weights $W^{(i)} = \bar{p}(X^{(i)})/q(X^{(i)})$. Note $\sum_i W^{(i)}/N \to Z$ by LLN. So:
$$\hat{I}_{\text{SNIS}} = \frac{\sum_i W^{(i)}\varphi(X^{(i)})}{\sum_j W^{(j)}} = \sum_i w^{(i)}\varphi(X^{(i)}),$$
where $w^{(i)} = W^{(i)}/\sum_j W^{(j)}$ are the normalised weights. **This estimator is biased** (ratio of two estimators), but consistent.

---

### D5. IS estimator for marginal likelihood $p(y)$ *(2025 Q2a, 2023 Q2)*

**Question.** Given $p(y) = \int p(y|x)p(x)dx$ with $p(x) = \bar{p}(x)/Z$ unknown. Derive a SNIS estimator.

**Answer.** Test function $\varphi(x) = p(y|x)$, target $p(x)$, so $p(y) = E_p[p(y|X)]$. Use $q(x)$ as proposal:
$$\hat{p}(y) = \frac{\sum_i W^{(i)} p(y|X^{(i)})}{\sum_j W^{(j)}}, \quad W^{(i)} = \frac{\bar{p}(X^{(i)})}{q(X^{(i)})}, \quad X^{(i)} \sim q.$$
This is **biased** (SNIS). A simple unbiased estimator exists only when $p$ is known: $\hat{p}(y) = \frac{1}{N}\sum_i p(y|X^{(i)})$ with $X^{(i)} \sim p$.

---
<a id='block-e'></a>
## Block E — Bayesian Conjugate Updates

---

### E1. Poisson-Gamma conjugacy *(2024 Q3b, 2023 Q3c)*

**Question.** Prior $p(x) = \text{Gamma}(x;\alpha,\beta)$, likelihood $p(y|x) = \text{Poisson}(y;x)$. Show the posterior is $\text{Gamma}(\alpha+y,\beta+1)$.

**Answer.**
$$p(x|y) \propto p(y|x)p(x) = \frac{x^y e^{-x}}{y!}\cdot\frac{\beta^\alpha}{\Gamma(\alpha)}x^{\alpha-1}e^{-\beta x} \propto x^{(\alpha+y)-1}e^{-(\beta+1)x}.$$
This is $\text{Gamma}(\alpha+y, \beta+1)$. $\square$

---

### E2. Sequential Bayesian update *(2024 Q3b)*

**Question.** For i.i.d. observations $y_{1:n}$ given $x$, prove:
$$p(x|y_{1:n}) = \frac{p(y_n|x)p(x|y_{1:n-1})}{p(y_n|y_{1:n-1})}.$$

**Answer.**
$$p(x|y_{1:n}) = \frac{p(y_{1:n}|x)p(x)}{p(y_{1:n})} = \frac{p(y_n|x)p(y_{1:n-1}|x)p(x)}{p(y_n|y_{1:n-1})p(y_{1:n-1})} = \frac{p(y_n|x)p(x|y_{1:n-1})}{p(y_n|y_{1:n-1})}.\quad\square$$

---

### E3. Sequential update recursion for sufficient statistics *(2024 Q3b)*

**Question.** $p(x) = \text{Gamma}(\alpha_0,\beta_0)$, $p(y_i|x) = \text{Poisson}(y_i;x)$. Derive recursion for $\alpha_n, \beta_n$.

**Answer.** By E1 applied sequentially:
$$p(x|y_{1:n}) = \text{Gamma}\!\left(\alpha_0 + \sum_{i=1}^n y_i,\ \beta_0 + n\right).$$
Recursion: $\alpha_n = \alpha_{n-1} + y_n$, $\beta_n = \beta_{n-1} + 1$.

---
<a id='block-f'></a>
## Block F — Metropolis-Hastings

---

### F1. MH pseudocode — independent, general, symmetric proposals *(every year)*

**Independent proposal $q(x')$:**
```
initialise X_0
for n = 1, 2, ...:
    sample X' ~ q(x')
    alpha = min(1, p*(X') * q(X_{n-1}) / (p*(X_{n-1}) * q(X')))
    with probability alpha: X_n = X'
    else: X_n = X_{n-1}
```

**General proposal $q(x'|x)$:**
```
initialise X_0
for n = 1, 2, ...:
    sample X' ~ q(x' | X_{n-1})
    alpha = min(1, p*(X') * q(X_{n-1}|X') / (p*(X_{n-1}) * q(X'|X_{n-1})))
    with probability alpha: X_n = X'
    else: X_n = X_{n-1}
```

**Symmetric proposal** $q(x'|x) = q(x|x')$: acceptance simplifies to $\alpha = \min(1, p^\star(X')/p^\star(X_{n-1}))$.

---

### F2. MH rejection probability at a point *(2025 Q3a)*

**Question.** The chain is at $x$. What is the probability the next proposed sample is **rejected**?

**Answer.** Averaging over the proposal:
$$P(\text{reject}|X_n = x) = \int q(x'|x)\left(1 - \min\!\left(1, \frac{p^\star(x')q(x|x')}{p^\star(x)q(x'|x)}\right)\right)dx'.$$

---

### F3. Detailed balance proof for MH *(2025 Q3a — 5 marks)*

**Question.** Prove MH satisfies detailed balance: $K(x'|x)p^\star(x) = K(x|x')p^\star(x')$.

**Answer.** The MH transition kernel (ignoring the self-loop term for $x \neq x'$) is:
$$K(x'|x) = q(x'|x)\cdot\alpha(x,x') = q(x'|x)\min\!\left(1, \frac{p^\star(x')q(x|x')}{p^\star(x)q(x'|x)}\right).$$

WLOG assume $p^\star(x)q(x'|x) \geq p^\star(x')q(x|x')$ (the case is symmetric). Then:
$$\alpha(x,x') = \frac{p^\star(x')q(x|x')}{p^\star(x)q(x'|x)}, \quad \alpha(x',x) = 1.$$

Check detailed balance:
$$K(x'|x)p^\star(x) = q(x'|x)\cdot\frac{p^\star(x')q(x|x')}{p^\star(x)q(x'|x)}\cdot p^\star(x) = p^\star(x')q(x|x') = K(x|x')p^\star(x').\quad\square$$

---

### F4. Simulated annealing (SA) for optimisation *(2025 Q3b)*

**Question.** Describe a Metropolis-based algorithm to globally optimise $f: \mathbb{R} \to \mathbb{R}$.

**Answer.** Target $p_T(x) \propto e^{f(x)/T}$ for temperature $T > 0$. Run MH targeting this:
```
initialise X_0, choose decreasing schedule T_n -> 0
for n = 1, 2, ...:
    sample X' ~ q(X' | X_{n-1})
    alpha = min(1, exp((f(X') - f(X_{n-1})) / T_n))
    with probability alpha: X_n = X'
    else: X_n = X_{n-1}
```
As $T_n \to 0$, $p_{T_n}$ concentrates on the global maximum.

---
<a id='block-g'></a>
## Block G — Langevin Algorithms

---

### G1. ULA and MALA pseudocode *(2023 Q3b, 2024 Q3a, 2025 Q3)*

**ULA** (Unadjusted Langevin Algorithm):
$$X_{n+1} = X_n + \gamma\nabla\log p^\star(X_n) + \sqrt{2\gamma}\,Z_n, \quad Z_n \sim \mathcal{N}(0,I).$$

```
initialise X_0, choose step size gamma > 0
for n = 0, 1, 2, ...:
    Z_n ~ N(0, I)
    X_{n+1} = X_n + gamma * grad_log_p*(X_n) + sqrt(2*gamma) * Z_n
```

**MALA** (Metropolis-Adjusted Langevin Algorithm): use ULA step as **proposal** inside MH:
$$q(x'|x) = \mathcal{N}\!\left(x' ; x + \gamma\nabla\log p^\star(x), 2\gamma I\right).$$
Accept/reject with $\alpha = \min(1, p^\star(x')q(x|x')/(p^\star(x)q(x'|x)))$.

**Key difference:** ULA has bias in the stationary distribution (does NOT target $p^\star$ exactly). MALA targets $p^\star$ exactly but requires an accept/reject step.

---

### G2. ULA limiting distribution *(2025 Q3c — 5 marks)*

**Question.** Modified ULA: $X_{k+1} = X_k + \gamma\nabla\log p^\star(X_k) + \sigma_q W_k$, $W_k \sim \mathcal{N}(0,1)$. Target $p^\star(x) = \mathcal{N}(x;\mu,\sigma^2)$. Find the limiting distribution.

**Answer.** $\nabla\log p^\star(x) = -(x-\mu)/\sigma^2$. The update is:
$$X_{k+1} = X_k - \frac{\gamma}{\sigma^2}(X_k - \mu) + \sigma_q W_k = \left(1 - \frac{\gamma}{\sigma^2}\right)X_k + \frac{\gamma\mu}{\sigma^2} + \sigma_q W_k.$$
This is an AR(1) process: $X_{k+1} = aX_k + b + \sigma_q W_k$ where $a = 1 - \gamma/\sigma^2$. For stationarity, need $|a| < 1$, i.e. $0 < \gamma < 2\sigma^2$.

At stationarity $X_\infty \sim \mathcal{N}(m, v^2)$, matching moments:
- Mean: $m = am + b \Rightarrow m(1-a) = b \Rightarrow m = b/(1-a) = \frac{\gamma\mu/\sigma^2}{\gamma/\sigma^2} = \mu$. ✓
- Variance: $v^2 = a^2 v^2 + \sigma_q^2 \Rightarrow v^2 = \frac{\sigma_q^2}{1-a^2}$.

So the limiting distribution is $\mathcal{N}\!\left(\mu,\ \dfrac{\sigma_q^2}{1-(1-\gamma/\sigma^2)^2}\right)$.

For ULA ($\sigma_q = \sqrt{2\gamma}$): $v^2 = \frac{2\gamma}{1-(1-\gamma/\sigma^2)^2} \neq \sigma^2$ in general — ULA is **biased**.

To make unbiased: set $v^2 = \sigma^2$, i.e. $\sigma_q^2 = \sigma^2(1 - (1-\gamma/\sigma^2)^2) = \sigma^2 \cdot \frac{\gamma}{\sigma^2}(2 - \gamma/\sigma^2) = \gamma(2 - \gamma/\sigma^2)$. So $\sigma_q = \sqrt{\gamma(2 - \gamma/\sigma^2)}$.

---

### G3. MALA proposal for Gaussian target *(2023 Q3b)*

**Question.** Derive the MALA proposal $q(x'|x)$ for $p^\star(x) = \mathcal{N}(x;\mu,\sigma^2)$.

**Answer.** $\nabla\log p^\star(x) = -(x-\mu)/\sigma^2$. MALA proposal:
$$q(x'|x) = \mathcal{N}\!\left(x';\ x - \frac{\gamma}{\sigma^2}(x-\mu),\ 2\gamma\right).$$

---

### G4. Stochastic gradient ULA (mini-batch) *(2024 Q3a, 2023 Q3)*

**Question.** Likelihood $p(y_{1:n}|x) = \prod_{i=1}^n p(y_i|x)$, $n$ very large. Modify ULA.

**Answer.** Stochastic Gradient ULA (SGULA): at each step, randomly subsample a mini-batch $S \subset \{1,\ldots,n\}$ of size $m$:
$$X_{k+1} = X_k + \gamma\left(\nabla\log p(X_k) + \frac{n}{m}\sum_{i\in S}\nabla\log p(y_i|X_k)\right) + \sqrt{2\gamma}Z_k.$$
This approximates the full gradient at $O(m)$ cost instead of $O(n)$.

---
<a id='block-h'></a>
## Block H — Fisher's Identity, EM, SOUL

---

### H1. Fisher's identity *(2025 Q2b — 3 marks, high value)*

**Question.** Prove:
$$\nabla_\theta\log p_\theta(y) = E_{p_\theta(x|y)}[\nabla_\theta\log p_\theta(x,y)].$$

**Answer.** Start from $p_\theta(y) = \int p_\theta(x,y)dx$. Differentiating (swap $\nabla$ and $\int$):
$$\nabla_\theta p_\theta(y) = \int \nabla_\theta p_\theta(x,y)dx.$$
Divide both sides by $p_\theta(y)$:
$$\nabla_\theta\log p_\theta(y) = \frac{\nabla_\theta p_\theta(y)}{p_\theta(y)} = \int \frac{\nabla_\theta p_\theta(x,y)}{p_\theta(y)}dx.$$
Multiply and divide inside by $p_\theta(x,y)$:
$$= \int \frac{p_\theta(x,y)}{p_\theta(y)}\cdot\frac{\nabla_\theta p_\theta(x,y)}{p_\theta(x,y)}dx = \int p_\theta(x|y)\nabla_\theta\log p_\theta(x,y)\,dx = E_{p_\theta(x|y)}[\nabla_\theta\log p_\theta(x,y)].\quad\square$$

---

### H2. IS estimator for the gradient (SOUL-style) *(2025 Q2b)*

**Question.** Using a proposal $q_\theta(x|y)$, derive a SNIS estimator for $\nabla_\theta\log p_\theta(y)$.

**Answer.** By Fisher's identity, we need $E_{p_\theta(x|y)}[\nabla_\theta\log p_\theta(x,y)]$. Use SNIS with $q_\theta$ as proposal:
$$\widehat{\nabla_\theta\log p_\theta(y)} = \sum_{i=1}^N w^{(i)}\nabla_\theta\log p_\theta(X^{(i)},y), \quad X^{(i)} \sim q_\theta(\cdot|y),$$
where $W^{(i)} = p_\theta(X^{(i)},y)/q_\theta(X^{(i)}|y)$ and $w^{(i)} = W^{(i)}/\sum_j W^{(j)}$.

---

### H3. EM algorithm *(Chapter 5)*

**Setup.** Maximise $\log p_\theta(y)$ when $x$ is latent. EM iterates:

**E-step:** Compute $Q(\theta, \theta_k) = E_{p_{\theta_k}(x|y)}[\log p_\theta(x,y)]$.

**M-step:** $\theta_{k+1} = \arg\max_\theta Q(\theta, \theta_k)$.

**Ascent property:** $\log p_{\theta_{k+1}}(y) \geq \log p_{\theta_k}(y)$. Proof sketch: use $\log p_\theta(y) = Q(\theta,\theta_k) - H(\theta,\theta_k)$ where $H(\theta,\theta_k) = E_{p_{\theta_k}(x|y)}[\log p_{\theta_k}(x|y)/p_\theta(x|y)] \geq 0$ by Jensen.

---
<a id='block-i'></a>
## Block I — State-Space Models & Bootstrap Particle Filter

---

### I1. SSM setup and filtering problem *(every year)*

**Model:**
$$X_0 \sim \mu(x_0), \quad X_t|X_{t-1} \sim f(x_t|x_{t-1}), \quad Y_t|X_t \sim g(y_t|x_t).$$

**Filtering problem:** compute $p(x_t|y_{1:t})$ recursively.

**Predict-update recursion:**
$$p(x_t|y_{1:t-1}) = \int f(x_t|x_{t-1})p(x_{t-1}|y_{1:t-1})dx_{t-1} \quad \text{(predict)}$$
$$p(x_t|y_{1:t}) = \frac{g(y_t|x_t)p(x_t|y_{1:t-1})}{p(y_t|y_{1:t-1})} \quad \text{(update)}.$$

**Incremental likelihood:**
$$p(y_t|y_{1:t-1}) = \int g(y_t|x_t)p(x_t|y_{1:t-1})dx_t.$$

---

### I2. Bootstrap Particle Filter (BPF) pseudocode *(every year — 3 marks)*

**Pseudocode (memorise this cold):**
```
// Initialisation t=0:
for i = 1, ..., N:
    sample x_0^(i) ~ mu(x_0)
    set W_0^(i) = 1/N

// For t = 1, ..., T:
for t = 1, ..., T:
    // Resample
    resample indices a^(i) ~ Categorical(W_{t-1}^(1), ..., W_{t-1}^(N))
    set x_{t-1}^(i) <- x_{t-1}^(a^(i))   // overwrite with resampled particles
    
    // Propagate
    for i = 1, ..., N:
        sample x_t^(i) ~ f(x_t | x_{t-1}^(i))
    
    // Reweight
    for i = 1, ..., N:
        set W_t^(i) = g(y_t | x_t^(i))
    normalise: W_t^(i) <- W_t^(i) / sum_j W_t^(j)
```

**Filtering approximation:** $p(x_t|y_{1:t}) \approx \sum_i W_t^{(i)}\delta_{x_t^{(i)}}$.

---

### I3. Log marginal likelihood in BPF (numerically stable) *(2025 Q4a — 4 marks)*

**Question.** Extend BPF to compute $\log p(y_{1:T})$ stably.

**Answer.** Incremental likelihood at step $t$: $p(y_t|y_{1:t-1}) \approx \frac{1}{N}\sum_i g(y_t|x_t^{(i)})$ (before normalisation). So:
$$\log p(y_{1:T}) = \sum_{t=1}^T \log p(y_t|y_{1:t-1}) \approx \sum_{t=1}^T \log\left(\frac{1}{N}\sum_{i=1}^N g(y_t|x_t^{(i)})\right).$$

**Numerical stability (log-sum-exp):** compute $\log\left(\frac{1}{N}\sum_i e^{\ell^{(i)}}\right)$ where $\ell^{(i)} = \log g(y_t|x_t^{(i)})$ via:
$$\ell_{\max} = \max_i \ell^{(i)}, \quad \log\left(\frac{1}{N}\sum_i e^{\ell^{(i)}}\right) = \ell_{\max} + \log\left(\frac{1}{N}\sum_i e^{\ell^{(i)} - \ell_{\max}}\right).$$
This avoids overflow/underflow by subtracting $\ell_{\max}$ before exponentiating.

**Add to BPF:**
```
log_ml = 0
// Inside BPF loop at time t, before normalisation:
    log_w = [log g(y_t | x_t^(i)) for i=1..N]
    l_max = max(log_w)
    log_ml += l_max + log(mean(exp(log_w - l_max)))
```

---
<a id='block-j'></a>
## Block J — SIR Smoother & Gradient Ascent for $\theta$

---

### J1. SIR smoother pseudocode *(2025 Q4b — 2 marks)*

**Setup.** We want to approximate $p_\theta(x_{0:T}|y_{1:T})$ (joint smoothing distribution).

**Pseudocode:**
```
// Initialisation:
for i = 1, ..., N:
    sample x_0^(i) ~ mu_theta(x_0)
    store full path: X_{0:0}^(i) = [x_0^(i)]
    set W_0^(i) = 1/N

// For t = 1, ..., T:
for t = 1, ..., T:
    // Resample PATHS
    resample indices a^(i) ~ Categorical(W_{t-1}^(1), ..., W_{t-1}^(N))
    X_{0:t-1}^(i) <- X_{0:t-1}^(a^(i))   // resample entire paths
    
    // Propagate (use transition as proposal)
    for i = 1, ..., N:
        sample x_t^(i) ~ f_theta(x_t | x_{t-1}^(i))
        append: X_{0:t}^(i) = [X_{0:t-1}^(i), x_t^(i)]
    
    // Reweight
    for i = 1, ..., N:
        W_t^(i) = g_theta(y_t | x_t^(i))
    normalise: W_t^(i) <- W_t^(i) / sum_j W_t^(j)

// Output: weighted paths {(X_{0:T}^(i), W_T^(i))}
```

---

### J2. Gradient of log marginal likelihood *(2025 Q4b — 4 marks)*

**Question.** Derive $\nabla_\theta\log p_\theta(y_{1:T})$ as an expectation.

**Answer.** Apply Fisher's identity with latent $x_{0:T}$:
$$\nabla_\theta\log p_\theta(y_{1:T}) = E_{p_\theta(x_{0:T}|y_{1:T})}[\nabla_\theta\log p_\theta(x_{0:T},y_{1:T})].$$

Now $p_\theta(x_{0:T},y_{1:T}) = \mu_\theta(x_0)\prod_{t=1}^T f_\theta(x_t|x_{t-1})g_\theta(y_t|x_t)$, so:
$$\nabla_\theta\log p_\theta(x_{0:T},y_{1:T}) = \nabla_\theta\log\mu_\theta(x_0) + \sum_{t=1}^T\left[\nabla_\theta\log f_\theta(x_t|x_{t-1}) + \nabla_\theta\log g_\theta(y_t|x_t)\right].$$

**SNIS estimator using SIR smoother output:**
$$\widehat{\nabla_\theta\log p_\theta(y_{1:T})} = \sum_{i=1}^N W_T^{(i)}\nabla_\theta\log p_\theta(X_{0:T}^{(i)}, y_{1:T}).$$

---

### J3. Gradient ascent for $\theta$ *(2025 Q4b — 3 marks)*

**Pseudocode:**
```
initialise theta_0, choose step size eta > 0
for k = 0, 1, 2, ...:
    run SIR smoother with current theta_k to get {(X_{0:T}^(i), W_T^(i))}
    compute gradient:
        g_k = sum_i W_T^(i) * grad_theta log p_{theta_k}(X_{0:T}^(i), y_{1:T})
    update:
        theta_{k+1} = theta_k + eta * g_k
```

This is the **SMOKE** / particle gradient ascent algorithm. Each outer iteration requires running a full SIR smoother.

---
## Quick-Reference Formula Sheet

| Formula | Expression |
|---------|------------|
| Inversion sampler | $X = F^{-1}(U)$, $U \sim \text{Unif}(0,1)$ |
| Rejection sampling bound | $p(x) \leq Mq(x)$, acceptance $= 1/M$ |
| Unnormalised rejection | acceptance $= Z/\bar{M}$ |
| Accepted samples distribution | $N \sim \text{Binomial}(K, 1/M)$ |
| MC estimator | $\hat{\varphi}^N = \frac{1}{N}\sum_i \varphi(X^{(i)})$, $X^{(i)} \sim p$ |
| MC variance | $\frac{1}{N}(\int\varphi^2 p\,dx - (\int\varphi p\,dx)^2)$ |
| IS estimator | $\hat{\varphi}^N_{\text{IS}} = \frac{1}{N}\sum_i w(X^{(i)})\varphi(X^{(i)})$, $w(x) = p(x)/q(x)$ |
| IS variance | $\frac{1}{N}(\int \varphi^2 p^2/q\,dx - I^2)$ |
| SNIS estimator | $\sum_i \tilde{w}^{(i)}\varphi(X^{(i)})$, $\tilde{w}^{(i)} = W^{(i)}/\sum_j W^{(j)}$ |
| MH acceptance | $\alpha = \min(1, p^\star(x')q(x|x')/(p^\star(x)q(x'|x)))$ |
| ULA update | $X_{n+1} = X_n + \gamma\nabla\log p^\star(X_n) + \sqrt{2\gamma}Z_n$ |
| AR(1) stationary variance | $\sigma^2/(1-a^2)$ |
| Fisher's identity | $\nabla_\theta\log p_\theta(y) = E_{p_\theta(x|y)}[\nabla_\theta\log p_\theta(x,y)]$ |
| BPF weight update | $W_t^{(i)} \propto g(y_t|x_t^{(i)})$ (bootstrap: propagate under $f$) |
| Log marginal likelihood | $\log p(y_{1:T}) = \sum_t\log\frac{1}{N}\sum_i g(y_t|x_t^{(i)})$ |
| Log-sum-exp trick | $\ell_{\max} + \log\frac{1}{N}\sum_i e^{\ell^{(i)}-\ell_{\max}}$ |